In [39]:
import math
import locale
import numpy as np
import mip
from collections import namedtuple
import glob
import pandas as pd
from IPython.display import display, Markdown

In [40]:
##### Open the main sheet #####

directory = "examples/Mariners 2026"
globFilename = directory + "/Full Name*.xls*"
excelFiles = glob.glob(globFilename)
if len(excelFiles) < 1 or len(excelFiles) > 1:
    print(f"Can't find unique excel file: {globFilename}")
    exit()
mainSheet = pd.read_excel(excelFiles[0])

In [41]:
##### Find the constraint row #####
for constraintRow in range(len(mainSheet.Date)):
    if mainSheet.iat[constraintRow,2] == "At least":
        break

In [42]:
##### Establish an object for each day of the season
 
locale.setlocale(locale.LC_ALL, '')
Game = namedtuple('Game', ('weekday', 'date', 'gameDay', 'time', 'opponent', 'type', 'price', 'pairs', 'seats'))

##### Read the season schedule and store it as a list of games
 
openingDay = mainSheet.Date[0]
schedule = []
GamesInPlan = 0
PairsInPlan = 0
MaxPairsPerGame = 0
gamesPerMonth = {}
totalMonths = 0
for weekday, date, time, opponent, type, price, pairs, seats in zip(mainSheet.Day, mainSheet.Date, mainSheet.Time, mainSheet.Opponent, mainSheet.Type, mainSheet.Price, mainSheet.GamePairs, mainSheet.Seats):
    if not isinstance(weekday, str) or weekday == "" or pd.isna(date):
        break
    schedule.append(Game(weekday, date, (date - openingDay).days, time.strftime("%I:%M %p"), opponent, type, price, pairs, seats))
    GamesInPlan += 1
    PairsInPlan += pairs
    MaxPairsPerGame = max(MaxPairsPerGame, pairs)
    if date.month not in gamesPerMonth:
        gamesPerMonth[date.month] = 1
        totalMonths += 1
    else:
        gamesPerMonth[date.month] += 1

# If a month has a standard deviation fewer games than average, assign it to a neighoring month
meanGamesPerMonth = 0
for gameCount in gamesPerMonth.values():
    meanGamesPerMonth += gameCount
meanGamesPerMonth /= totalMonths
varGamesPerMonth = 0
for gameCount in gamesPerMonth.values():
    varGamesPerMonth += (gameCount - meanGamesPerMonth) ** 2
sigma = math.sqrt(varGamesPerMonth / totalMonths)
logicalMonths = {} 
for month, gameCount in gamesPerMonth.items():
    if gameCount > meanGamesPerMonth - sigma:
        logicalMonths[month] = month
for month in gamesPerMonth.keys():
    if month not in logicalMonths:
        # First, try to join a later month
        for newMonth in range(month + 1, totalMonths):
            if newMonth in logicalMonths:
                logicalMonths[month] = newMonth
                break
    if month not in logicalMonths:
        # Otherwise, join an earlier month
        for newMonth in range(month - 1, -1, -1):
            if newMonth in logicalMonths:
                logicalMonths[month] = newMonth
                break

In [43]:
#########################################################
# Build a dictionary out of a list of games
#
#     buildCode == 0:    Create constraints for pair and quads
#     buildCode == 2:    Create constraint for pairs only
#     buildCode == 4:    Create constraint for quads only

def BuildDict(gameList, buildCode = 0):
    pairList = []
    for game in gameList:
        if buildCode != 4:
            pairList.append((game, 1.0))
        if buildCode != 2:
            pairList.append((game + GamesInPlan, 1.0))
    return dict(pairList)

# Define a class to contain constraints

class Constraint:
    def __init__(self, description, isChecked, comparator, value, gameDictionaries = None):
        self.description = description
        self.isChecked = isChecked
        self.comparator = comparator
        self.value = value
        self.gameDictionaries = list() if gameDictionaries is None else gameDictionaries

# Define a class to contain each person's preferences

class SportsFan:
    def __init__(self, name, pairs, quads, ranking, extra = None):
        self.name = name
        self.pairs = pairs
        self.quads = quads

# Assign weights to games

        if len(ranking) != 0:
            self.ranking = ranking
        else:
            self.ranking = GamesInPlan * [GamesInPlan // 2]

# Adjust weights to favor highly ranked games

        self.useRanking = []
        midpoint = (GamesInPlan - 1) // 2
        for wgt in self.ranking:
            if wgt <= midpoint + 1:
                self.useRanking.append(math.sqrt(wgt - 1.0))
            else:
                self.useRanking.append(2.0 * math.sqrt(midpoint) - math.sqrt(2.0 * midpoint - wgt + 1))
        if len(self.useRanking) == GamesInPlan:
            self.useRanking += [2.0 * cost for cost in self.useRanking]
        else:
            for ix in range(GamesInPlan):
                self.useRanking[GamesInPlan + ix] *= 2

# Each person must attend correct number of games

        pairCon = Constraint(f"{self.name}: Pairs = {self.pairs}", True, '==', self.pairs, [dict([(ix, 1.0) for ix in range(GamesInPlan)])])
        quadCon = Constraint(f"{self.name}: Quads = {self.quads}", True, '==', self.quads, [dict([(ix, 1.0) for ix in range(GamesInPlan, 2 * GamesInPlan)])])

# Save all of the constraints for this person

        self.constraints = [pairCon, quadCon]
        if extra is not None:
            self.constraints += extra

##### This constraint handles the spacing of games

def Spacing(description, checked, comparator, value, pairsOrQuads = 0):
    constraint = Constraint(description, checked, comparator, np.float64(1))
    if not isinstance(value, (int,float,np.floating,np.integer)) or pd.isna(value) or value < 1 or value > GamesInPlan:
        print(f"Bad constraint:{description}")
        return constraint
    daysApart = value + 1
    firstGame = 0
    lastGame = 0
    while True:
        while lastGame < GamesInPlan and schedule[firstGame].gameDay + daysApart > schedule[lastGame].gameDay:
            lastGame += 1
        constraint.gameDictionaries.append(BuildDict(range(firstGame, lastGame), pairsOrQuads))
        if lastGame == GamesInPlan:
            break
        while schedule[firstGame].gameDay + daysApart <= schedule[lastGame].gameDay:
            firstGame += 1
    return constraint

##### Require or forbid games in various months

def Monthly(description, checked, comparator, value, pairsOrQuads = 0):
    constraint = Constraint(description, checked, comparator, value)
    if not isinstance(value, (int,float,np.floating,np.integer)) or pd.isna(value) or value < 1 or value > 30:
        print(f"Bad constraint:{description}")
        return constraint
    firstGame = 0
    month = logicalMonths[schedule[firstGame].date.month]
    lastGame = 0
    while True:
        while lastGame < GamesInPlan and logicalMonths[schedule[lastGame].date.month] == month:
            lastGame += 1
        constraint.gameDictionaries.append(BuildDict(range(firstGame, lastGame), pairsOrQuads))
        if lastGame == GamesInPlan:
            break
        firstGame = lastGame
        month = logicalMonths[schedule[firstGame].date.month]
    return constraint

##### Require or forbid games in different series

def Series(description, checked, comparator, value, pairsOrQuads = 0):
    constraint = Constraint(description, checked, comparator, value)
    if not isinstance(value, (int,float,np.floating,np.integer)) or pd.isna(value) or value < 1 or value > GamesInPlan:
        print(f"Bad constraint:{description}")
        return constraint
    firstGame = 0
    lastGame = 0
    while True:
        while lastGame < GamesInPlan and schedule[firstGame].opponent == schedule[lastGame].opponent:
            lastGame += 1
        constraint.gameDictionaries.append(BuildDict(range(firstGame, lastGame), pairsOrQuads))
        if lastGame == GamesInPlan:
            break
        firstGame = lastGame
    return constraint

##### Require or forbid games for different opponents

def Opponents(description, checked, comparator, value, pairsOrQuads = 0):
    constraint = Constraint(description, checked, comparator, value)
    if not isinstance(value, (int,float,np.floating,np.integer)) or pd.isna(value) or value < 1 or value > GamesInPlan:
        print(f"Bad constraint:{description}")
        return constraint
    firstGame = 0
    lastGame = 0
    opponentDict = {}
    while True:
        while lastGame < GamesInPlan and schedule[firstGame].opponent == schedule[lastGame].opponent:
            lastGame += 1
        opponentGames = opponentDict.get(schedule[firstGame].opponent, [])
        opponentGames += range(firstGame, lastGame)
        opponentDict[schedule[firstGame].opponent] = opponentGames
        if lastGame == GamesInPlan:
            break
        firstGame = lastGame
    for opponentGames in opponentDict.values():
        constraint.gameDictionaries.append(BuildDict(opponentGames, pairsOrQuads))
    return constraint


In [44]:
SpreadSheetConstraint = namedtuple('SpreadSheetConstraint', ('function', 'comparator', 'pairsOrQuads'))
constraintsToProcess = [SpreadSheetConstraint(Monthly, '>=', 0),
                 SpreadSheetConstraint(Monthly, '<=', 0),
                 SpreadSheetConstraint(Monthly, '>=', 2),
                 SpreadSheetConstraint(Monthly, '<=', 2),
                 SpreadSheetConstraint(Monthly, '>=', 4),
                 SpreadSheetConstraint(Monthly, '<=', 4),
                 SpreadSheetConstraint(Spacing, '<=', 0),
                 SpreadSheetConstraint(Spacing, '>=', 0),
                 SpreadSheetConstraint(Spacing, '<=', 2),
                 SpreadSheetConstraint(Spacing, '>=', 2),
                 SpreadSheetConstraint(Spacing, '<=', 4),
                 SpreadSheetConstraint(Spacing, '>=', 4),
                 SpreadSheetConstraint(Series, '<=', 0),
                 SpreadSheetConstraint(Opponents, '<=', 0)]

In [45]:
##### Define the participants here #####

fans = []
totalPairs = 0
for fullName, nPairs, nQuads in zip(mainSheet.FullName, mainSheet.Pairs, mainSheet.Quads):
    if not isinstance(fullName, str) or fullName == "":
        break
    totalPairs += nPairs + 2 * nQuads
    globFilename = directory + "/" + fullName + "*.xls*"
    excelFiles = glob.glob(globFilename)
    if len(excelFiles) < 1 or len(excelFiles) > 1:
        print(f"Can't find unique excel file: {globFilename}")
        continue
    fanSheet = pd.read_excel(excelFiles[0])
    pairsRank = [np.int64(fanSheet.Pick[ix]) for ix in range(GamesInPlan)]
    if not isinstance(fanSheet.QuadPick[0], np.float64) and not math.isnan(fanSheet.QuadPick[0]):
        quadsRank = [np.int64(fanSheet.QuadPick[ix]) for ix in range(GamesInPlan)]
        pairsRank += quadsRank

    # Add fan constraints
    extraConstraints = []
    for ix, sheetConstraint in enumerate(constraintsToProcess):
        row = constraintRow + ix
        if fanSheet.iat[row,1] or not pd.isna(fanSheet.iat[row,3]):
            description = f"{fullName}: {fanSheet.iat[row,2]} {fanSheet.iat[row,3]} {fanSheet.iat[row,4]}"
            extraConstraints.append(sheetConstraint.function(description, fanSheet.iat[row,1], sheetConstraint.comparator, fanSheet.iat[row,3], sheetConstraint.pairsOrQuads))
    fans.append(SportsFan(fullName, nPairs, nQuads, pairsRank, extraConstraints))

leftOver = PairsInPlan - totalPairs
if leftOver < 0:
    print("Too many games requested")
maxSparePairs = max((leftOver * GamesInPlan) // PairsInPlan, 1) # Maximum # of games that could be completely unassigned
while leftOver > 0:
    pairsRank = (GamesInPlan * [np.int64(1)])[:]
    nPairs = min(leftOver, maxSparePairs)
    fans.append(SportsFan(f"Spare Pair", nPairs, 0, pairsRank))
    leftOver -= nPairs


In [46]:
for gix, game in enumerate(schedule):
    picks = []
    for fan in fans:
        picks.append(int(fan.ranking[gix]))
    print(f"{picks} {game.weekday} {game.date.strftime('%x')} {game.opponent}{game.time} {game.type} {game.seats}")

[81, 81, 52, 1, 67, 40, 2, 39, 56, 81, 80, 50, 1] Thu 3/26/2026 Guardians 07:10 PM A $90.00 (4)
[80, 80, 51, 54, 66, 45, 41, 40, 55, 80, 22, 49, 19] Fri 3/27/2026 Guardians 06:45 PM B $69.00 (4)
[79, 79, 32, 3, 65, 81, 68, 1, 54, 79, 4, 1, 20] Sat 3/28/2026 Guardians 06:40 PM B $69.00 (4)
[78, 78, 57, 53, 64, 80, 58, 42, 80, 78, 58, 48, 21] Sun 3/29/2026 Guardians 04:20 PM B $69.00 (4)
[77, 17, 56, 52, 63, 41, 42, 41, 53, 77, 12, 56, 52] Mon 3/30/2026 Yankees 06:40 PM D $45.00 (4)
[76, 18, 78, 51, 78, 79, 74, 43, 52, 76, 40, 10, 51] Tue 3/31/2026 Yankees 06:40 PM D $45.00 (4)
[75, 19, 77, 50, 7, 78, 3, 44, 66, 75, 72, 57, 76] Wed 4/1/2026 Yankees 01:10 PM D $45.00 (4)
[74, 77, 25, 35, 54, 47, 65, 2, 26, 27, 3, 80, 40] Fri 4/10/2026 Astros 06:40 PM D $45.00 (4)
[73, 76, 14, 81, 77, 77, 70, 3, 25, 18, 10, 81, 41] Sat 4/11/2026 Astros 06:40 PM D $45.00 (4)
[72, 75, 26, 34, 79, 21, 56, 45, 78, 17, 39, 47, 69] Sun 4/12/2026 Astros 01:10 PM D $45.00 (4)
[71, 74, 7, 33, 11, 76, 63, 46, 64, 34

In [47]:
try:
    tixModel = mip.Model()
    tixVars = []
    for fan in fans:
        tixVars += [tixModel.add_var(name = fan.name + f"_pair_game_{ix}", var_type = mip.BINARY) for ix in range(GamesInPlan)]
        tixVars += [tixModel.add_var(name = fan.name + f"_quad_game_{ix}", var_type = mip.BINARY) for ix in range(GamesInPlan)]
    slackVars = [tixModel.add_var(name = f"slack_game_{ix}+", var_type = mip.CONTINUOUS, ub = 0.0) for ix in range(GamesInPlan)]
    slackVars += [tixModel.add_var(name = f"slack_game_{ix}-", var_type = mip.CONTINUOUS, ub = 0.0) for ix in range(GamesInPlan)]

    # All tickets must be allocated

    for ix in range(GamesInPlan):
        tixModel.add_constr(mip.xsum(tixVars[ix + 2 * iy * GamesInPlan] + 2.0 * tixVars[ix + GamesInPlan + 2 * iy * GamesInPlan] for iy in range(len(fans))) + slackVars[2 * ix] - slackVars[2 * ix + 1] == schedule[ix].pairs, name = f"Allocate_All_Tickets_game_{ix}")

    # Each fan must attend the correct number of games + satisfy all personal constraints

    for iy, fan in enumerate(fans):
        for ix, constraint in enumerate(fan.constraints):
            for coefDict in constraint.gameDictionaries:
                slackVars += [tixModel.add_var(name = f"slack_{constraint.description}+", var_type = mip.CONTINUOUS, ub = 0.0)]
                slackVars += [tixModel.add_var(name = f"slack_{constraint.description}-", var_type = mip.CONTINUOUS, ub = 0.0)]
                linFunc = mip.xsum(count * tixVars[iz + iy * 2 * GamesInPlan] for iz, count in coefDict.items()) + slackVars[-2] - slackVars[-1]
                if constraint.comparator == '==':
                    tixModel.add_constr(linFunc == constraint.value, name = f"{constraint.description}")
                if constraint.comparator == '<=':
                    tixModel.add_constr(linFunc <= constraint.value, name = f"{constraint.description}")
                if constraint.comparator == '>=':
                    tixModel.add_constr(linFunc >= constraint.value, name = f"{constraint.description}")

    # Establish the objective function

    costs = []
    for fan in fans:
        costs += fan.useRanking
    tixModel.objective = mip.xsum(costs[ix] * tixVars[ix] for ix in range(len(tixVars))) + mip.xsum(100000.0 * slackVars[ix] for ix in range(len(slackVars)))
except Exception as e:
    print(f"Mip setup exception: {e}")

In [48]:
try:
    status = tixModel.optimize()
except Exception as e:
    print(f"Mip optimize exception: {e}")

if status != mip.OptimizationStatus.OPTIMAL:
    print(f"No solution found: {status}")
    if status == mip.OptimizationStatus.INFEASIBLE:
        print("Infeasible solution found.  Relaxing constraints to find a solution with minimum slack.")
        tixRelax = tixModel.copy()
        for var in tixRelax.vars:
            if var.name.startswith("slack_"):
                var.ub = 1.0
        try:
            status = tixRelax.optimize(max_seconds = 60)
        except Exception as e:
            print(f"Mip relax optimize exception: {e}")
        print(status)
        violatedConstraints = []
        for var in tixRelax.vars:
            if var.name.startswith("slack") and (var.x is None or var.x > 0.0):
                violatedConstraints.append(var.name.removeprefix("slack_").removesuffix("+").removesuffix("-"))
        print("Violated constraints:  ", violatedConstraints)
        for constraint in tixRelax.constrs:
            if constraint.name in violatedConstraints:
                print(constraint)

No solution found: OptimizationStatus.INFEASIBLE
Infeasible solution found.  Relaxing constraints to find a solution with minimum slack.
OptimizationStatus.OPTIMAL
Violated constraints:   ['Ryan Falls: At least 1.0 game(s) per month (pairs)']
Ryan Falls: At least 1.0 game(s) per month (pairs): +1.0 Ryan Falls_pair_game_0 +1.0 Ryan Falls_pair_game_1 +1.0 Ryan Falls_pair_game_2
	 +1.0 Ryan Falls_pair_game_3 +1.0 Ryan Falls_pair_game_4 +1.0 Ryan Falls_pair_game_5
	 +1.0 Ryan Falls_pair_game_6 +1.0 Ryan Falls_pair_game_7 +1.0 Ryan Falls_pair_game_8
	 +1.0 Ryan Falls_pair_game_9 +1.0 Ryan Falls_pair_game_10 +1.0 Ryan Falls_pair_game_11
	 +1.0 Ryan Falls_pair_game_12 +1.0 Ryan Falls_pair_game_13 +1.0 Ryan Falls_pair_game_14
	 +1.0 Ryan Falls_pair_game_15 +1.0 Ryan Falls_pair_game_16 +1.0 slack_Ryan Falls: At least 1.0 game(s) per month (pairs)+
	 -1.0 slack_Ryan Falls: At least 1.0 game(s) per month (pairs)- >= 1.0
Ryan Falls: At least 1.0 game(s) per month (pairs): +1.0 Ryan Falls_pair_game

In [49]:
picks = [[] for fan in fans]
for mix, mvar in enumerate(tixModel.vars):
    if mvar.x is not None and mvar.x > 0.5:
        fix = mix // (2 * GamesInPlan)
        if len(fans[fix].ranking) > GamesInPlan:
            gix = mix % (2 * GamesInPlan)
        else:
            gix = mix % GamesInPlan
        picks[fix].append(fans[fix].ranking[gix])
for fix, fan in enumerate(fans):
    picks[fix].sort()
    picks[fix] = [int(pick) for pick in picks[fix]]
    print(fans[fix].name, picks[fix])

Al Erisman []
Andrew Erisman []
Cole Graves []
Eric Brechner []
Fritz Klein []
Jordan Bork []
Karen Boehmer []
Michael Erisman []
Michael Jones []
Michael Rochester []
Mike Neff []
Ryan Falls []
Tom Grandine []


In [50]:
costs = {}
gameAllocationTable = "Day | Date | Time | Opponent | Type | Seats |"
for ix in range(MaxPairsPerGame):
    gameAllocationTable += f" Pair{ix+1} |"
gameAllocationTable += "\n| :-: | :-: | :-: | :-: | :-: | :-: |"  + " :-: |" * MaxPairsPerGame + "\n"
for gix, game in enumerate(schedule):
    gameAllocationTable += f"| {game.weekday} | {game.date.strftime('%x')} | {game.time} | {game.opponent} | {game.type} | {game.seats} |"
    for mix, mvar in enumerate(tixModel.vars):
        if mix % GamesInPlan == gix and mvar.x is not None and mvar.x > 0.5:
            fix = mix // (2 * GamesInPlan)
            if len(fans[fix].ranking) > GamesInPlan and mix % (2 * GamesInPlan) >= GamesInPlan:
                gix += GamesInPlan
            cost = costs.get(fans[fix].name, 0.0)
            costs[fans[fix].name] = cost + 2.0 * game.price
            gameAllocationTable += f" {fans[fix].name} ({fans[fix].ranking[gix]}) |"
            if mix % (2 * GamesInPlan) >= GamesInPlan:
                costs[fans[fix].name] += 2.0 * game.price
                gameAllocationTable += " |"
    gameAllocationTable += " ❌ |" * (MaxPairsPerGame - game.pairs) + "\n"
display(Markdown(gameAllocationTable))

Day | Date | Time | Opponent | Type | Seats | Pair1 | Pair2 |
| :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
| Thu | 3/26/2026 | 07:10 PM | Guardians  | A | $90.00 (4) |
| Fri | 3/27/2026 | 06:45 PM | Guardians  | B | $69.00 (4) |
| Sat | 3/28/2026 | 06:40 PM | Guardians  | B | $69.00 (4) |
| Sun | 3/29/2026 | 04:20 PM | Guardians  | B | $69.00 (4) |
| Mon | 3/30/2026 | 06:40 PM | Yankees  | D | $45.00 (4) |
| Tue | 3/31/2026 | 06:40 PM | Yankees  | D | $45.00 (4) |
| Wed | 4/1/2026 | 01:10 PM | Yankees  | D | $45.00 (4) |
| Fri | 4/10/2026 | 06:40 PM | Astros  | D | $45.00 (4) |
| Sat | 4/11/2026 | 06:40 PM | Astros  | D | $45.00 (4) |
| Sun | 4/12/2026 | 01:10 PM | Astros  | D | $45.00 (4) |
| Mon | 4/13/2026 | 01:10 PM | Astros  | B | $69.00 (4) |
| Fri | 4/17/2026 | 06:40 PM | Rangers  | B | $69.00 (4) |
| Sat | 4/18/2026 | 04:15 PM | Rangers  | B | $69.00 (4) |
| Sun | 4/19/2026 | 01:10 PM | Rangers  | B | $69.00 (4) |
| Mon | 4/20/2026 | 06:40 PM | Athletics  | B | $69.00 (4) |
| Tue | 4/21/2026 | 06:40 PM | Athletics  | B | $69.00 (4) |
| Wed | 4/22/2026 | 01:10 PM | Athletics  | D | $45.00 (4) |
| Fri | 5/1/2026 | 06:45 PM | Royals  | D | $45.00 (4) |
| Sat | 5/2/2026 | 06:40 PM | Royals  | A | $90.00 (4) |
| Sun | 5/3/2026 | 01:10 PM | Royals  | A | $90.00 (4) |
| Mon | 5/4/2026 | 06:40 PM | Braves  | A | $90.00 (4) |
| Tue | 5/5/2026 | 06:40 PM | Braves  | B | $69.00 (4) |
| Wed | 5/6/2026 | 01:10 PM | Braves  | B | $69.00 (4) |
| Fri | 5/15/2026 | 06:40 PM | Padres  | B | $69.00 (4) |
| Sat | 5/16/2026 | 04:15 PM | Padres  | D | $45.00 (4) |
| Sun | 5/17/2026 | 04:20 PM | Padres  | D | $45.00 (4) |
| Mon | 5/18/2026 | 06:40 PM | White Sox  | D | $45.00 (4) |
| Tue | 5/19/2026 | 06:40 PM | White Sox  | B | $69.00 (4) |
| Wed | 5/20/2026 | 01:10 PM | White Sox  | B | $69.00 (4) |
| Fri | 5/29/2026 | 07:10 PM | D-backs  | B | $69.00 (4) |
| Sat | 5/30/2026 | 07:10 PM | D-backs  | C | $57.00 (4) |
| Sun | 5/31/2026 | 01:10 PM | D-backs  | C | $57.00 (4) |
| Mon | 6/1/2026 | 06:40 PM | Mets  | C | $57.00 (4) |
| Tue | 6/2/2026 | 06:40 PM | Mets  | A | $90.00 (4) |
| Wed | 6/3/2026 | 12:40 PM | Mets  | A | $90.00 (4) |
| Tue | 6/16/2026 | 06:40 PM | Orioles  | A | $90.00 (4) |
| Wed | 6/17/2026 | 06:40 PM | Orioles  | B | $69.00 (4) |
| Thu | 6/18/2026 | 01:10 PM | Orioles  | B | $69.00 (4) |
| Fri | 6/19/2026 | 07:10 PM | Red Sox  | B | $69.00 (4) |
| Sat | 6/20/2026 | 07:10 PM | Red Sox  | C | $57.00 (4) |
| Sun | 6/21/2026 | 01:10 PM | Red Sox  | C | $57.00 (4) |
| Mon | 6/29/2026 | 06:40 PM | Angels  | C | $57.00 (4) |
| Tue | 6/30/2026 | 06:40 PM | Angels  | B | $69.00 (4) |
| Thu | 7/2/2026 | 06:40 PM | Angels  | A | $90.00 (4) |
| Fri | 7/3/2026 | 07:10 PM | Blue Jays  | A | $90.00 (4) |
| Sat | 7/4/2026 | 01:10 PM | Blue Jays  | A | $90.00 (4) |
| Sun | 7/5/2026 | 02:00 PM | Blue Jays  | A | $90.00 (4) |
| Fri | 7/17/2026 | 07:10 PM | Giants  | A | $90.00 (4) |
| Sat | 7/18/2026 | 05:08 PM | Giants  | A | $90.00 (4) |
| Sun | 7/19/2026 | 01:10 PM | Giants  | C | $57.00 (4) |
| Mon | 7/20/2026 | 06:40 PM | Reds  | C | $57.00 (4) |
| Tue | 7/21/2026 | 06:40 PM | Reds  | C | $57.00 (4) |
| Wed | 7/22/2026 | 12:40 PM | Reds  | B | $69.00 (4) |
| Fri | 7/31/2026 | 07:10 PM | Twins  | A | $90.00 (4) |
| Sat | 8/1/2026 | 01:10 PM | Twins  | A | $90.00 (4) |
| Sun | 8/2/2026 | 01:10 PM | Twins  | A | $90.00 (4) |
| Tue | 8/4/2026 | 06:40 PM | Tigers  | C | $57.00 (4) |
| Wed | 8/5/2026 | 06:40 PM | Tigers  | C | $57.00 (4) |
| Thu | 8/6/2026 | 01:10 PM | Tigers  | C | $57.00 (4) |
| Fri | 8/7/2026 | 07:10 PM | Rays  | A | $90.00 (4) |
| Sat | 8/8/2026 | 06:50 PM | Rays  | A | $90.00 (4) |
| Sun | 8/9/2026 | 01:10 PM | Rays  | A | $90.00 (4) |
| Fri | 8/21/2026 | 07:10 PM | Cubs  | B | $69.00 (4) |
| Sat | 8/22/2026 | 06:40 PM | Cubs  | B | $69.00 (4) |
| Sun | 8/23/2026 | 01:10 PM | Cubs  | B | $69.00 (4) |
| Mon | 8/24/2026 | 06:40 PM | Phillies  | B | $69.00 (4) |
| Tue | 8/25/2026 | 06:40 PM | Phillies  | B | $69.00 (4) |
| Wed | 8/26/2026 | 01:10 PM | Phillies  | B | $69.00 (4) |
| Thu | 9/3/2026 | 06:40 PM | Athletics  | C | $57.00 (4) |
| Fri | 9/4/2026 | 07:10 PM | Athletics  | C | $57.00 (4) |
| Sat | 9/5/2026 | 06:40 PM | Athletics  | C | $57.00 (4) |
| Sun | 9/6/2026 | 01:10 PM | Athletics  | C | $57.00 (4) |
| Tue | 9/8/2026 | 06:40 PM | Rangers  | B | $69.00 (4) |
| Wed | 9/9/2026 | 06:40 PM | Rangers  | B | $69.00 (4) |
| Thu | 9/10/2026 | 01:10 PM | Rangers  | B | $69.00 (4) |
| Tue | 9/22/2026 | 06:40 PM | Astros  | C | $57.00 (4) |
| Wed | 9/23/2026 | 07:10 PM | Astros  | C | $57.00 (4) |
| Thu | 9/24/2026 | 06:40 PM | Angels  | C | $57.00 (4) |
| Fri | 9/25/2026 | 07:10 PM | Angels  | A | $90.00 (4) |
| Sat | 9/26/2026 | 06:40 PM | Angels  | A | $90.00 (4) |
| Sun | 9/27/2026 | 12:10 PM | Angels  | A | $90.00 (4) |


In [51]:
amountOwedTable = "| Full Name | Amount owed | Paid |\n"
amountOwedTable += "| :- | -: | -: |\n"
for name, cost in sorted(costs.items()):
    amountOwedTable += f"| {name} | {locale.currency(cost, grouping=True)} | |\n"
display(Markdown(amountOwedTable))

| Full Name | Amount owed | Paid |
| :- | -: | -: |
